# 04 — Multi-Model Comparison for Journey Marketing Tasks

**Foundations | Mastering Agentic AI for Customer Journey Marketing**

Compare GPT-4.1, Claude, and Gemini across four journey marketing tasks:
1. Intent scoring from raw signals
2. Email personalisation for nurture
3. Objection handling during decision
4. Churn prediction from usage data

All three providers use the OpenAI-compatible client pattern.

In [ ]:
# File      : 04_multi_model_comparison.ipynb
# Stage     : Foundations
# Chapter   : 2
# Framework : OpenAI / Anthropic / Google
# Author    : Pushparajan Ramar
# Repo      : https://github.com/Pushparajan/agenticai-marketing

import os, json, time
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
USE_MOCK = os.getenv("USE_MOCK_APIS", "true").lower() == "true"

# Configure providers using OpenAI-compatible interface
PROVIDERS = {}

if not USE_MOCK:
    if os.getenv("OPENAI_API_KEY"):
        PROVIDERS["GPT-4.1"] = {"client": OpenAI(), "model": "gpt-4.1"}
    if os.getenv("ANTHROPIC_API_KEY"):
        PROVIDERS["Claude"] = {
            "client": OpenAI(api_key=os.getenv("ANTHROPIC_API_KEY"), base_url="https://api.anthropic.com/v1/"),
            "model": "claude-sonnet-4-20250514"
        }

print(f"Mock mode: {USE_MOCK}")
print(f"Active providers: {list(PROVIDERS.keys()) if PROVIDERS else ['Mock (all 3 simulated)']}")

## Helper: Run the Same Prompt Across All Models

In [ ]:
def run_comparison(task_name: str, prompt: str) -> dict:
    """Run a prompt across all configured providers and compare results."""
    results = {}

    if USE_MOCK:  # MOCK MODE
        mock_responses = {
            "intent_scoring": {
                "GPT-4.1": '{"score": 67, "stage": "consideration", "signals": ["pricing_page", "case_study"]}',
                "Claude": '{"score": 72, "stage": "consideration", "signals": ["pricing_page", "case_study", "webinar"]}',
                "Gemini": '{"score": 65, "stage": "awareness", "signals": ["pricing_page", "blog"]}'
            },
            "email_personalisation": {
                "GPT-4.1": "Subject: Your pipeline forecast accuracy is leaving money on the table",
                "Claude": "Subject: How Acme Corp can improve forecast accuracy by 40%",
                "Gemini": "Subject: Revenue forecasting best practices for B2B SaaS teams"
            },
            "objection_handling": {
                "GPT-4.1": "I understand budget is a concern. Our customers see 3x ROI within 90 days.",
                "Claude": "Budget concerns are valid. Let me share how similar companies justified the investment.",
                "Gemini": "Many customers start with a pilot programme to prove ROI before full commitment."
            },
            "churn_prediction": {
                "GPT-4.1": '{"risk": "high", "probability": 0.73, "driver": "usage_decline", "action": "exec_outreach"}',
                "Claude": '{"risk": "high", "probability": 0.68, "driver": "usage_decline", "action": "product_training"}',
                "Gemini": '{"risk": "medium", "probability": 0.55, "driver": "feature_gap", "action": "roadmap_preview"}'
            }
        }
        for provider in ["GPT-4.1", "Claude", "Gemini"]:
            results[provider] = {
                "response": mock_responses.get(task_name, {}).get(provider, "[Mock]"),
                "latency_ms": {"GPT-4.1": 820, "Claude": 650, "Gemini": 740}[provider]
            }
    else:
        for name, cfg in PROVIDERS.items():
            start = time.time()
            resp = cfg["client"].chat.completions.create(
                model=cfg["model"],
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3
            )
            elapsed = (time.time() - start) * 1000
            results[name] = {"response": resp.choices[0].message.content, "latency_ms": round(elapsed)}

    return results

def print_comparison(task: str, results: dict):
    print(f"\n{'='*70}")
    print(f"TASK: {task}")
    print(f"{'='*70}")
    for provider, data in results.items():
        print(f"\n--- {provider} ({data['latency_ms']}ms) ---")
        print(data["response"])

## Task 1: Intent Scoring (Awareness Stage)

In [ ]:
intent_prompt = """Score this prospect's buying intent (0-100) and classify their journey stage.
Return JSON: {score, stage, signals}

Signals:
- Visited pricing page 2x this week
- Downloaded "Revenue Forecasting Guide" PDF
- Attended webinar on pipeline management
- Company: 180 employees, B2B SaaS, Series B
- No demo request yet"""

results = run_comparison("intent_scoring", intent_prompt)
print_comparison("Intent Scoring", results)

## Task 2: Email Personalisation (Consideration Stage)

In [ ]:
email_prompt = """Write a personalised email subject line for this prospect.
Name: Alex Rivera, VP Sales at Acme Corp (B2B SaaS, 180 employees)
Interests: revenue forecasting, pipeline accuracy
Last action: downloaded comparison guide 3 days ago
Goal: get them to book a demo"""

results = run_comparison("email_personalisation", email_prompt)
print_comparison("Email Personalisation", results)

## Task 3: Objection Handling (Decision Stage)

In [ ]:
objection_prompt = """The prospect says: 'The budget is tight this quarter and we need to see
clear ROI before committing to a $48K annual contract.'

Product: Revenue Intelligence platform. Average customer sees 3x ROI in 90 days.
Write a concise, empathetic response that addresses the budget concern."""

results = run_comparison("objection_handling", objection_prompt)
print_comparison("Objection Handling", results)

## Task 4: Churn Prediction (Retention Stage)

In [ ]:
churn_prompt = """Analyse this customer's churn risk. Return JSON: {risk, probability, driver, action}

Customer data:
- Active features: dropped from 8 to 3 over 30 days
- Login frequency: down 60%
- NPS: was 8, now 5
- Support tickets: 4 in last 2 weeks (all about reporting bugs)
- Contract renewal: 45 days away
- Competitor Clari mentioned in last support call"""

results = run_comparison("churn_prediction", churn_prompt)
print_comparison("Churn Prediction", results)

## Summary Comparison Table

In [ ]:
print(f"{'Task':<25} {'GPT-4.1':>10} {'Claude':>10} {'Gemini':>10}")
print("-" * 57)

tasks = [
    ("Intent Scoring", "intent_scoring"),
    ("Email Personalisation", "email_personalisation"),
    ("Objection Handling", "objection_handling"),
    ("Churn Prediction", "churn_prediction")
]

# Latency comparison (mock values)
for name, key in tasks:
    r = run_comparison(key, "")
    latencies = [f"{r[p]['latency_ms']}ms" for p in ["GPT-4.1", "Claude", "Gemini"]]
    print(f"{name:<25} {latencies[0]:>10} {latencies[1]:>10} {latencies[2]:>10}")

## Key Takeaways

1. **All three models** handle journey marketing tasks well with the OpenAI-compatible client
2. **GPT-4.1** excels at structured JSON output and tool calling
3. **Claude** produces more nuanced, empathetic copy
4. **Gemini** offers competitive pricing for high-volume tasks like intent scoring
5. Use **Ollama** with local models for zero-cost development and testing

The choice of model depends on the stage: high-volume Awareness tasks benefit from
cheaper models, while Decision and Retention tasks benefit from higher quality.

**Next:** Stage 1 — Awareness with OpenAI Agents SDK →